# COMP5318 Assignment 1: Rice Classification

##### Group number: 92
##### Student 1 SID: 550455842
##### Student 2 SID: ...  
##### Student 3 SID: ... 

## **1. Data Pre-processing**

### 1.1 Libraries

In [93]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from warnings import simplefilter

### 1.2 Ignore future warnings

In [94]:
simplefilter(action='ignore', category=FutureWarning)

### 1.3 Read dataset

In [95]:
rice_dataset = pd.read_csv('rice-final2.csv', na_values='?')

### 1.4 Filling in the missing attribute values 

#### 1.4.1 Check values

In [96]:
rice_dataset.head()

,Area,Perimiter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,class
0,12573.0,461.466003,192.903351,84.572075,0.898772,12893.0,0.550433,class2
1,12845.0,464.121002,194.332214,85.524338,0.897952,13125.0,0.774962,class2
2,14055.0,488.748993,207.751755,87.250328,0.907536,14484.0,0.550076,class1
3,14412.0,490.324005,207.476135,89.689514,0.901735,14703.0,0.598853,class1
4,14658.0,477.117004,189.566635,99.997780,0.849551,15048.0,0.649504,class2


#### 1.4.2 Check null values of each columns

In [97]:
rice_dataset.isnull().sum()

Area                 4
Perimiter            4
Major_Axis_Length    5
Minor_Axis_Length    3
Eccentricity         6
Convex_Area          5
Extent               2
class                0
dtype: int64

#### 1.4.3 Filling in the missing attribute values

In [98]:
imputer = SimpleImputer(strategy='mean')

feature_cols = rice_dataset.columns.drop('class')

rice_imputed = rice_dataset.copy()

rice_imputed[feature_cols] = imputer.fit_transform(rice_dataset[feature_cols])

In [99]:
rice_imputed.isnull().sum()

Area                 0
Perimiter            0
Major_Axis_Length    0
Minor_Axis_Length    0
Eccentricity         0
Convex_Area          0
Extent               0
class                0
dtype: int64

### 1.5 Normalising	the	data

#### 1.5.1 Check min and max values

In [100]:
rice_imputed[feature_cols].min()

Area                 7943.000000
Perimiter             359.100006
Major_Axis_Length     145.264465
Minor_Axis_Length      63.344753
Eccentricity            0.799511
Convex_Area          8080.000000
Extent                  0.511160
dtype: float64

In [101]:
rice_imputed[feature_cols].max()

Area                 17948.000000
Perimiter              548.445984
Major_Axis_Length      238.435089
Minor_Axis_Length      107.542450
Eccentricity             0.934006
Convex_Area          18322.000000
Extent                   0.839661
dtype: float64

#### 1.5.2 normalising

In [102]:
scaler = MinMaxScaler()

feature_cols_imputed = rice_imputed.columns.drop('class')

rice_normalised_imputed = rice_imputed.copy()

rice_normalised_imputed[feature_cols_imputed] = scaler.fit_transform(rice_imputed[feature_cols_imputed])

#### 1.5.3 Check min and max values again

In [103]:
rice_normalised_imputed[feature_cols_imputed].min()

Area                 0.0
Perimiter            0.0
Major_Axis_Length    0.0
Minor_Axis_Length    0.0
Eccentricity         0.0
Convex_Area          0.0
Extent               0.0
dtype: float64

In [104]:
rice_normalised_imputed[feature_cols_imputed].max()

Area                 1.0
Perimiter            1.0
Major_Axis_Length    1.0
Minor_Axis_Length    1.0
Eccentricity         1.0
Convex_Area          1.0
Extent               1.0
dtype: float64

### 1.6 Changing the class values

#### 1.6.1 Check values

In [105]:
rice_normalised_imputed['class'].unique()

array(['class2', 'class1'], dtype=object)

#### 1.6.2 Mapping

In [106]:
rice_encoded_normalised_imputed = rice_normalised_imputed.copy()

rice_encoded_normalised_imputed['class'] = rice_encoded_normalised_imputed['class'].map({
    'class1': 0,
    'class2': 1
})

#### 1.6.3 Check values again

In [107]:
rice_encoded_normalised_imputed['class'].unique()

array([1, 0])

### 1.7 Print the first	10 rows of the pre-processed dataset

In [108]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])

In [109]:
X = rice_encoded_normalised_imputed.drop(columns=['class']).to_numpy()
y = rice_encoded_normalised_imputed['class'].to_numpy()

print(X, y)

[[0.46276862 0.54062937 0.511308   ... 0.73802366 0.46992775 0.11955211]
 [0.48995502 0.55465132 0.52664399 ... 0.73192824 0.49257957 0.80304787]
 [0.61089455 0.68472005 0.67067587 ... 0.80318891 0.6252685  0.11846508]
 ...
 [0.69975012 0.80824013 0.88173494 ... 0.96167029 0.69000195 0.91686252]
 [0.28835582 0.29972122 0.23711135 ... 0.49679629 0.30706893 0.43393018]
 [0.73893053 0.79053698 0.7942185  ... 0.8393834  0.73413396 0.20958212]] [1 1 0 ... 0 1 0]


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### 2.1 Preparation

#### 2.1.1 Libraries

In [110]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.svm import SVC

#### 2.1.2 CV K-fold

In [111]:
cvKFold=StratifiedKFold(
    n_splits=10, 
    shuffle=True, 
    random_state=0
)

#### 2.1.3 Train/test split

In [112]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=0
)

### **2.2 Part 1: Cross-validation without parameter tuning**

#### 2.2.2 Logistic Regression

In [113]:
# Logistic Regression
lr = LogisticRegression(
    max_iter=1000,
    random_state=0
)

lr_scores = cross_val_score(
    lr,
    X,
    y,
    cv=cvKFold,
    scoring='accuracy'
)

lr_cv_accuracy = lr_scores.mean()

In [114]:
# Naïve Bayes
nb = GaussianNB()

nb_scores = cross_val_score(
    nb,
    X,
    y,
    cv=cvKFold,
    scoring='accuracy'
)

nb_cv_accuracy = nb_scores.mean()

#### 2.2.3 Part 1 Results


In [115]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {lr_cv_accuracy:.4f}")
print(f"NB average cross-validation accuracy: {nb_cv_accuracy:.4f}")

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### **2.3 Part 2: Cross-validation with parameter tuning**

#### 2.3.1 KNN

In [116]:
# KNN 
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]

knn = KNeighborsClassifier()

knn_param_grid = {
    'n_neighbors': k,
    'p': p
}

knn_grid = GridSearchCV(
    estimator=knn,
    param_grid=knn_param_grid,
    scoring='accuracy',
    cv=cvKFold
)

knn_grid.fit(X_train, y_train)

knn_pred = knn_grid.predict(X_test)

knn_test_accuracy = accuracy_score(y_test, knn_pred)

In [117]:
# Decision Tree 
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

dt = DecisionTreeClassifier(random_state=0)

dt_param_grid = {
    'max_depth': max_depth,
    'min_samples_split': min_samples_split,
    'min_samples_leaf': min_samples_leaf
}

dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param_grid,
    scoring='accuracy',
    cv=cvKFold
)

dt_grid.fit(X_train, y_train)

dt_pred = dt_grid.predict(X_test)

dt_test_accuracy = accuracy_score(y_test, dt_pred)

In [118]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

ada_param_grid = {
    'n_estimators': n_estimators,
    'learning_rate': learning_rate
}

ada_grid = GridSearchCV(
    AdaBoostClassifier(random_state=0),
    ada_param_grid,
    scoring='accuracy',
    cv=cvKFold
)

ada_grid.fit(X_train, y_train)
ada_pred = ada_grid.predict(X_test)
ada_test_accuracy = accuracy_score(y_test, ada_pred)

In [119]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

gb_param_grid = {
    'n_estimators': n_estimators,
    'learning_rate': learning_rate,
    'max_depth': max_depth
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=0),
    gb_param_grid,
    scoring='accuracy',
    cv=cvKFold
)

gb_grid.fit(X_train, y_train)
gb_pred = gb_grid.predict(X_test)
gb_test_accuracy = accuracy_score(y_test, gb_pred)


In [120]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to ‘sqrt’.
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]

rf_param_grid = {
    'n_estimators': n_estimators,
    'max_leaf_nodes': max_leaf_nodes
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=0),
    rf_param_grid,
    scoring='accuracy',
    cv=cvKFold
)

rf_grid.fit(X_train, y_train)
rf_pred = rf_grid.predict(X_test)

rf_test_accuracy = accuracy_score(y_test, rf_pred)
rf_macro_f1 = f1_score(y_test, rf_pred, average='macro')
rf_weighted_f1 = f1_score(y_test, rf_pred, average='weighted')

In [121]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
# optional
kernel = ['linear', 'rbf']

svm_param_grid = {
    'C': C,
    'kernel': kernel,
    'gamma': gamma
}

svm_grid = GridSearchCV(
    SVC(),
    svm_param_grid,
    scoring='accuracy',
    cv=cvKFold
)

svm_grid.fit(X_train, y_train)
svm_pred = svm_grid.predict(X_test)
svm_test_accuracy = accuracy_score(y_test, svm_pred)


#### 2.3.2 Part 2: Results

In [122]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# example printing:
print("KNN best k: ", knn_grid.best_params_['n_neighbors'])
print("KNN best p: ", knn_grid.best_params_['p'])
print(f"KNN cross-validation accuracy: {knn_grid.best_score_:.4f}")
print(f"KNN test set accuracy: {knn_test_accuracy:.4f}")
print()

print("Decision Tree best parameters:", dt_grid.best_params_)
print(f"Decision Tree cross-validation accuracy: {dt_grid.best_score_:.4f}")
print(f"Decision Tree test set accuracy: {dt_test_accuracy:.4f}")
print()

print("AdaBoost best parameters:", ada_grid.best_params_)
print(f"AdaBoost cross-validation accuracy: {ada_grid.best_score_:.4f}")
print(f"AdaBoost test set accuracy: {ada_test_accuracy:.4f}")
print()

print("Gradient Boosting best parameters:", gb_grid.best_params_)
print(f"Gradient Boosting cross-validation accuracy: {gb_grid.best_score_:.4f}")
print(f"Gradient Boosting test set accuracy: {gb_test_accuracy:.4f}")
print()

print("RF best n_estimators: ", rf_grid.best_params_['n_estimators'])
print("RF best max_leaf_nodes: ", rf_grid.best_params_['max_leaf_nodes'])
print(f"RF cross-validation accuracy: {rf_grid.best_score_:.4f}")
print(f"RF test set accuracy: {rf_test_accuracy:.4f}")
print(f"RF test set macro average F1: {rf_macro_f1:.4f}")
print(f"RF test set weighted average F1: {rf_weighted_f1:.4f}")
print()

print("SVM best parameters:", svm_grid.best_params_)
print(f"SVM cross-validation accuracy: {svm_grid.best_score_:.4f}")
print(f"SVM test set accuracy: {svm_test_accuracy:.4f}")

KNN best k:  7
KNN best p:  2
KNN cross-validation accuracy: 0.9375
KNN test set accuracy: 0.9250

Decision Tree best parameters: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
Decision Tree cross-validation accuracy: 0.9357
Decision Tree test set accuracy: 0.9429

AdaBoost best parameters: {'learning_rate': 0.2, 'n_estimators': 150}
AdaBoost cross-validation accuracy: 0.9455
AdaBoost test set accuracy: 0.9429

Gradient Boosting best parameters: {'learning_rate': 0.1, 'max_depth': 1, 'n_estimators': 50}
Gradient Boosting cross-validation accuracy: 0.9446
Gradient Boosting test set accuracy: 0.9429

RF best n_estimators:  10
RF best max_leaf_nodes:  6
RF cross-validation accuracy: 0.9402
RF test set accuracy: 0.9429
RF test set macro average F1: 0.9415
RF test set weighted average F1: 0.9428

SVM best parameters: {'C': 5, 'gamma': 1, 'kernel': 'rbf'}
SVM cross-validation accuracy: 0.9429
SVM test set accuracy: 0.9321


### Test your code

In [125]:
#load the test dataset to test out your model 

## **3. Reflection and Discussion**



## **AI Acknowledgement**